# 🏦 UPI Remitter Bank Reliability Analysis: Bank Mapping

🎯 **Goal:** Give every bank one standard name across all 24 months, so we can follow a bank over time.

📥 **Input:** `upi_remitter_clean.csv` (1,200 rows from notebook 01)

📤 **Output:** `bank_mapping.csv` (raw name to standard name, with reasons) and `upi_remitter_mapped.csv` (1,198 rows)

🪜 **Steps:** load, profile names, clean spellings, find banks that appear or vanish, test the suspects, build the mapping, apply it, merge duplicates, check, save

## Step 1: Load the clean table from notebook 01

**What:** Upload `upi_remitter_clean.csv` and read it.



In [1]:
import pandas as pd
import numpy as np
import re
from google.colab import files

files.upload()   # choose upi_remitter_clean.csv
df = pd.read_csv('upi_remitter_clean.csv', parse_dates=['month'])
print(df.shape)
df.head()

Saving upi_remitter_clean.csv to upi_remitter_clean.csv
(1200, 6)


,month,bank_raw,volume_mn,approved_pct,bd_pct,td_pct
0,2024-08-01,State Bank Of India,4052.31,94.40,5.39,0.21
1,2024-08-01,HDFC Bank Ltd.,1270.59,95.28,4.66,0.06
2,2024-08-01,Bank of Baroda,985.86,92.45,6.95,0.60
3,2024-08-01,Union Bank Of India,941.93,93.51,5.83,0.65
4,2024-08-01,Punjab national Bank,835.00,91.99,7.92,0.09


## Step 2: Profile the bank names

**What:** Count how many different names appear.

**Why:** There are only about 50 banks per month, so a large number of names means the same bank is written in several ways.



In [3]:
print('rows:', len(df))
print('months:', df.month.nunique())
print('distinct bank names:', df.bank_raw.nunique())
print('banks per month:', sorted(set(df.groupby('month').bank_raw.nunique().tolist()))) #"Banks per month" shows 49 or 50. A 49 means a name appears twice in one month, which we find in Step 5.

rows: 1200
months: 24
distinct bank names: 117
banks per month: [49, 50]


## Step 3: Group names that differ only in spelling

**What:** Make a "key" for each name: lowercase, no dots or commas, no "Ltd/Limited/Private/The". Names with the same key are the same bank.

**Why:** `State Bank Of India` and `State Bank of India` should not count as two banks.



In [4]:
def name_key(name):
    k = name.lower().replace('&', 'and')
    k = re.sub(r'[.,]', '', k)
    k = re.sub(r'\b(ltd|limited|private|pvt|the)\b', '', k)
    return re.sub(r'[^a-z0-9]', '', k)

df['key'] = df.bank_raw.map(name_key)
print('distinct names:', df.bank_raw.nunique(), '->', df.key.nunique(), 'after cleaning')

spellings = df.groupby('key').bank_raw.unique()
spellings[spellings.str.len() > 1].head(8)

distinct names: 117 -> 67 after cleaning


,bank_raw
key,
airtelpaymentsbank,"[Airtel Payments Bank Ltd, Airtel Payments Ban..."
ausmallfinancebank,"[AU small Finance Bank, AU Small Finance Bank,..."
axisbank,"[Axis bank Ltd., Axis Bank Ltd.]"
bandhanbank,"[Bandhan Bank limited, Bandhan Bank Limited, B..."
bankofbaroda,"[Bank of Baroda, Bank Of Baroda]"
bankofindia,"[Bank Of India, Bank of India]"
bankofmaharashtra,"[Bank of Maharashtra, Bank Of Maharashtra]"
centralbankofindia,"[Central Bank of India, Central Bank Of India]"


## Step 4: Which banks appear in which months?

**What:** For each bank, find its first month, last month, and how many months it appears in.

**Why:** A bank that appears late, vanishes, or lasts only a few months could be a new entry, an exit, or a rename. Spelling cleanup cannot find these.



In [5]:
presence = df.groupby('key').agg(
    first_month=('month', 'min'),
    last_month=('month', 'max'),
    months=('month', 'nunique'),
    rows=('month', 'size'))
presence['example_name'] = df.groupby('key').bank_raw.agg(lambda s: s.value_counts().index[0])

print('banks in all 24 months:', (presence.months == 24).sum())
print('banks not in all 24 months:', (presence.months < 24).sum())

partial = presence[presence.months < 24].sort_values('first_month').copy()
partial['first_month'] = partial.first_month.dt.strftime('%Y-%m')
partial['last_month'] = partial.last_month.dt.strftime('%Y-%m')
partial[['example_name', 'first_month', 'last_month', 'months', 'rows']]

banks in all 24 months: 38
banks not in all 24 months: 29


,example_name,first_month,last_month,months,rows
key,,,,,
equitassmallfinancebank,Equitas Small Finance Bank,2024-08,2026-07,23,23
tamilnadmercantilebank,Tamilnad Mercantile Bank Limited,2024-08,2026-07,23,23
karnatakagraminbankerstwhilepragathikrishnagraminbank,Karnataka Gramin Bank (Erstwhile Pragathi Kris...,2024-08,2024-08,1,1
ujjivansmallfinancebank,Ujjivan Small Finance Bank Limited,2024-08,2026-07,23,23
idfcfirstbank,IDFC First Bank Ltd,2024-08,2024-09,2,2
sarvaharyanagraminbank,Sarva Haryana Gramin Bank,2024-08,2026-07,22,22
rblbank,RBL Bank Ltd.,2024-08,2026-04,21,21
standardcharteredbank,Standard Chartered Bank,2024-08,2025-10,13,13
rajasthanmarudharagraminbank,Rajasthan Marudhara Gramin Bank,2024-08,2026-07,21,21


## Step 5: Find a name listed twice in one month

**What:** Show banks where `rows` is bigger than `months`.

**Why:** If a bank has more rows than months, NPCI listed it twice in some month. Joining on bank name later would double-count it.



In [7]:
presence[presence.rows > presence.months][['example_name', 'months', 'rows']]

,example_name,months,rows
key,,,
slicesmallfinancebank,Slice Small Finance Bank Limited,10,12


## Step 6: Test banks that seem to replace each other

**What:** Three pairs look suspicious: one ends exactly where another starts (Baroda U.P. and Uttar Pradesh Gramin, the two Rajasthan names, the two Andhra names). We print their volumes side by side.

**Why:** If it is one bank under a new name, volume should continue smoothly across the gap.


In [8]:
def compare(words, start, end):
    pattern = '|'.join(words)
    sub = df[df.bank_raw.str.lower().str.contains(pattern)]
    sub = sub[(sub.month >= start) & (sub.month <= end)]
    return sub.pivot_table(index='month', columns='bank_raw', values='volume_mn', aggfunc='sum').round(1)

checks = [
    ('Baroda U.P. vs Uttar Pradesh Gramin', ['baroda u', 'uttar pradesh'], '2025-08-01', '2026-02-01'),
    ('Rajasthan Marudhara vs Rajasthan Gramin', ['rajasthan'], '2025-08-01', '2026-02-01'),
    ('Andhra Pragathi vs Andhra Pradesh Grameena', ['andhra pragathi', 'andhra pradesh grameena bank'], '2025-07-01', '2026-01-01'),
]
for title, words, start, end in checks:
    print(title)
    display(compare(words, start, end))

Baroda U.P. vs Uttar Pradesh Gramin


bank_raw,Baroda U.P. Bank,Uttar Pradesh Gramin Bank
month,,
2025-08-01,53.8,NaN
2025-09-01,54.0,NaN
2025-10-01,NaN,61.4
2025-11-01,NaN,65.1
2025-12-01,NaN,71.4
2026-02-01,75.3,NaN


Rajasthan Marudhara vs Rajasthan Gramin


bank_raw,Rajasthan Gramin Bank,Rajasthan Marudhara Gramin Bank
month,,
2025-08-01,NaN,32.7
2025-09-01,NaN,42.8
2025-10-01,47.7,NaN
2025-11-01,49.2,NaN
2025-12-01,54.9,NaN
2026-01-01,NaN,57.4
2026-02-01,NaN,56.2


Andhra Pragathi vs Andhra Pradesh Grameena


bank_raw,Andhra Pradesh Grameena Bank,Andhra Pragathi Grameena Bank
month,,
2025-07-01,NaN,31.9
2025-08-01,NaN,32.1
2025-09-01,NaN,31.5
2025-11-01,31.7,NaN
2025-12-01,39.7,NaN


## Step 7: More evidence for the merges (and one against)

**What:** Two checks.
1. The three Andhra names in 2024. If two names appear in the same month, they are different banks.
2. Early renames (IDFC, Karnataka Gramin, Baroda U.P.) to see if volume continues.

**Why:** Merge only what the data supports.


In [9]:
print('Andhra, 2024')
display(compare(['andhra'], '2024-08-01', '2025-01-01'))
print('IDFC')
display(compare(['idfc'], '2024-08-01', '2024-11-01'))
print('Karnataka Gramin')
display(compare(['karnataka gramin', 'pragathi krishna'], '2024-08-01', '2024-11-01'))
print('Baroda U.P.')
display(compare(['baroda u'], '2024-08-01', '2024-12-01'))

Andhra, 2024


bank_raw,Andhra Pradesh Grameena Vikas Bank,Andhra Pradesh Grameena Vikas Bank - IMPS,Andhra Pragathi Grameena Bank
month,,,
2024-08-01,NaN,28.6,17.8
2024-09-01,29.9,NaN,19.0
2024-10-01,32.9,NaN,21.9
2024-11-01,31.7,NaN,21.6
2024-12-01,29.8,NaN,26.1
2025-01-01,NaN,NaN,25.6


IDFC


bank_raw,IDFC Bank Ltd.,IDFC First Bank Ltd
month,,
2024-08-01,NaN,90.0
2024-09-01,NaN,89.7
2024-10-01,99.7,NaN
2024-11-01,93.1,NaN


Karnataka Gramin


bank_raw,Karnataka Gramin Bank,Karnataka Gramin Bank (Erstwhile Pragathi Krishna Gramin Bank),Pragathi Krishna Gramin Bank
month,,,
2024-08-01,NaN,30.2,NaN
2024-09-01,NaN,NaN,28.7
2024-10-01,30.9,NaN,NaN
2024-11-01,30.3,NaN,NaN


Baroda U.P.


bank_raw,Baroda U.P. Bank,Baroda U.P. Bank (Erstwhile Baroda Uttar Pradesh Gramin Bank),Baroda UP Gramin Bank
month,,,
2024-08-01,NaN,25.0,NaN
2024-09-01,NaN,NaN,26.5
2024-10-01,NaN,NaN,31.1
2024-11-01,29.7,NaN,NaN
2024-12-01,33.2,NaN,NaN


## Step 8: Start the mapping table with the spelling fixes

**What:** One row per raw name. The standard name is the most common spelling in its group (not ALL CAPS). Two names are fixed by hand: `Bank Of India` and `Bandhan Bank limited`.

**Why:** This is the lookup table we will apply to the data. `rule` records why each row was changed.



In [10]:
mapping = df[['bank_raw', 'key']].drop_duplicates().reset_index(drop=True)

def pick_standard(spellings):
    counts = df[df.bank_raw.isin(spellings)].bank_raw.value_counts()
    not_caps = [s for s in counts.index if not s.isupper()]
    return (not_caps or list(counts.index))[0]

mapping['standard_name'] = mapping.groupby('key').bank_raw.transform(lambda s: pick_standard(list(s)))
mapping['standard_name'] = mapping.standard_name.replace({
    'Bank Of India': 'Bank of India',
    'Bandhan Bank limited': 'Bandhan Bank Limited'})

group_size = mapping.groupby('key').bank_raw.transform('count')
mapping['rule'] = np.where(group_size > 1, 'spelling', 'unchanged')
mapping['is_ranked'] = 1
mapping['possible_same_as'] = ''
mapping['note'] = ''
mapping[mapping.rule == 'spelling'].sort_values('standard_name').head(10)

,bank_raw,key,standard_name,rule,is_ranked,possible_same_as,note
25,AU small Finance Bank,ausmallfinancebank,AU Small Finance Bank,spelling,1,,
108,Au Small Finance Bank,ausmallfinancebank,AU Small Finance Bank,spelling,1,,
78,AU Small Finance Bank,ausmallfinancebank,AU Small Finance Bank,spelling,1,,
75,Airtel Payments Bank Ltd.,airtelpaymentsbank,Airtel Payments Bank Limited,spelling,1,,
65,Airtel Payments Bank Limited,airtelpaymentsbank,Airtel Payments Bank Limited,spelling,1,,
11,Airtel Payments Bank Ltd,airtelpaymentsbank,Airtel Payments Bank Limited,spelling,1,,
6,Axis bank Ltd.,axisbank,Axis Bank Ltd.,spelling,1,,
53,Axis Bank Ltd.,axisbank,Axis Bank Ltd.,spelling,1,,
83,BANDHAN BANK LIMITED,bandhanbank,Bandhan Bank Limited,spelling,1,,
58,Bandhan Bank Limited,bandhanbank,Bandhan Bank Limited,spelling,1,,


## Step 9: Add our decisions on renames
**What:**

- **Merge** six names into their current bank. NPCI's own title says "erstwhile" for three; the other three (IDFC First and the Andhra IMPS entry) are name-style changes backed by continuous volume.
- **Keep separate** three entries that look like renames but are not proven. We record the likely match in `possible_same_as`.

**Why:** Merge what we can prove, and flag the rest. If an interviewer asks, every decision has a reason in the table.


In [11]:
RENAMES = {
    'IDFC First Bank Ltd': ('IDFC Bank Ltd.', 'Name style changed from Oct 2024, volume continues'),
    'Karnataka Gramin Bank (Erstwhile Pragathi Krishna Gramin Bank)': ('Karnataka Gramin Bank', 'NPCI title says erstwhile'),
    'Pragathi Krishna Gramin Bank': ('Karnataka Gramin Bank', 'Old name, per NPCI title'),
    'Baroda U.P. Bank (Erstwhile Baroda Uttar Pradesh Gramin Bank)': ('Baroda U.P. Bank', 'NPCI title says erstwhile'),
    'Baroda UP Gramin Bank': ('Baroda U.P. Bank', 'Old name, per NPCI title'),
    'Andhra Pradesh Grameena Vikas Bank - IMPS': ('Andhra Pradesh Grameena Vikas Bank', 'Only entry in Aug 2024, continues Sep 2024'),
}
for raw, (std, why) in RENAMES.items():
    assert raw in set(mapping.bank_raw), raw
    assert std in set(mapping.standard_name), std
    i = mapping.bank_raw == raw
    mapping.loc[i, 'standard_name'] = std
    mapping.loc[i, 'rule'] = 'renamed_merged'
    mapping.loc[i, 'note'] = why

SEPARATE = {
    'Uttar Pradesh Gramin Bank': 'Baroda U.P. Bank',
    'Rajasthan Gramin Bank': 'Rajasthan Marudhara Gramin Bank',
    'Andhra Pradesh Grameena Bank': 'Andhra Pragathi Grameena Bank',
}
for raw, other in SEPARATE.items():
    assert raw in set(mapping.bank_raw), raw
    i = mapping.bank_raw == raw
    mapping.loc[i, 'rule'] = 'kept_separate'
    mapping.loc[i, 'possible_same_as'] = other
    mapping.loc[i, 'note'] = 'Volume continues across the gap, likely same bank, not proven'

mapping[mapping.rule.isin(['renamed_merged', 'kept_separate'])][['bank_raw', 'standard_name', 'rule', 'possible_same_as']]

,bank_raw,standard_name,rule,possible_same_as
23,IDFC First Bank Ltd,IDFC Bank Ltd.,renamed_merged,
36,Karnataka Gramin Bank (Erstwhile Pragathi Kris...,Karnataka Gramin Bank,renamed_merged,
37,Andhra Pradesh Grameena Vikas Bank - IMPS,Andhra Pradesh Grameena Vikas Bank,renamed_merged,
43,Baroda U.P. Bank (Erstwhile Baroda Uttar Prade...,Baroda U.P. Bank,renamed_merged,
60,Pragathi Krishna Gramin Bank,Karnataka Gramin Bank,renamed_merged,
61,Baroda UP Gramin Bank,Baroda U.P. Bank,renamed_merged,
87,Uttar Pradesh Gramin Bank,Uttar Pradesh Gramin Bank,kept_separate,Baroda U.P. Bank
88,Rajasthan Gramin Bank,Rajasthan Gramin Bank,kept_separate,Rajasthan Marudhara Gramin Bank
90,Andhra Pradesh Grameena Bank,Andhra Pradesh Grameena Bank,kept_separate,Andhra Pragathi Grameena Bank


## Step 10: Mark entries that should not be ranked

**What:** Set `is_ranked = 0` for Tri O Tech and One Mobikwik (100% approved, zero declines) and the four credit card entries (not regular banks, one month each). Also add a note on Slice.

**Why:** They would distort rankings. They stay in the data and the totals.



In [12]:
unranked = mapping.bank_raw.str.contains('tri o tech|mobikwik|credit card|sbi cards', case=False)
mapping.loc[unranked, 'is_ranked'] = 0
mapping.loc[unranked, 'note'] = 'Not a regular bank or no declines; excluded from rankings'

is_slice = mapping.standard_name == 'Slice Small Finance Bank Limited'
mapping.loc[is_slice, 'note'] = 'Listed twice in Mar 2026 and Jun 2026; rows merged in a later step'

mapping[mapping.is_ranked == 0][['bank_raw', 'note']]

,bank_raw,note
29,Tri O Tech Solutions Private Limited,Not a regular bank or no declines; excluded fr...
82,Axis Bank Credit Card,Not a regular bank or no declines; excluded fr...
84,HDFC Bank Credit Card HCB,Not a regular bank or no declines; excluded fr...
85,SBI Cards and Payment Services Limited,Not a regular bank or no declines; excluded fr...
86,ICICI Bank Credit Card,Not a regular bank or no declines; excluded fr...
100,ONE MOBIKWIK SYSTEMS LIMITED,Not a regular bank or no declines; excluded fr...
116,One Mobikwik Systems Limited,Not a regular bank or no declines; excluded fr...


## Step 11: Check the mapping table

**What:** Make sure every raw name appears once and has a standard name, then count.

**Why:** A mapping with a gap would drop rows silently when applied.



In [13]:
assert mapping.bank_raw.is_unique
assert mapping.standard_name.notna().all()

print('raw names:', len(mapping))
print('standard names:', mapping.standard_name.nunique())
print('ranked banks:', mapping[mapping.is_ranked == 1].standard_name.nunique())
mapping.rule.value_counts()

raw names: 117
standard names: 61
ranked banks: 55


,count
rule,
spelling,85
unchanged,23
renamed_merged,6
kept_separate,3


## Step 12: Apply the mapping to the data

**What:** Join the mapping onto the 1,200 rows to add `standard_name`, then look for a bank listed twice in a month.

**Why:** The row count must stay at 1,200, so nothing is lost or duplicated by the join. Then we can see exactly which rows need merging.


In [14]:
df2 = df.merge(mapping[['bank_raw', 'standard_name', 'is_ranked']],
               on='bank_raw', how='left', validate='many_to_one')
assert len(df2) == len(df) == 1200
assert df2.standard_name.notna().all()
print('rows after mapping:', len(df2))

dups = df2[df2.duplicated(['month', 'standard_name'], keep=False)]
dups[['month', 'bank_raw', 'standard_name', 'volume_mn', 'td_pct']]

rows after mapping: 1200


,month,bank_raw,standard_name,volume_mn,td_pct
983,2026-03-01,Slice Small Finance Bank Limited,Slice Small Finance Bank Limited,70.17,0.27
985,2026-03-01,Slice Small Finance Bank Limited,Slice Small Finance Bank Limited,66.82,0.54
1131,2026-06-01,Slice Small Finance Bank Limited,Slice Small Finance Bank Limited,90.55,0.04
1137,2026-06-01,Slice Small Finance Bank Limited,Slice Small Finance Bank Limited,65.27,0.31


## Step 13: Merge the duplicate Slice rows

**What:** For each bank and month, add the volumes and average the rates, weighted by volume.

**Why:** A big entry should count more than a small one, and this keeps estimated failed payments exactly the same as before. The table goes from 1,200 to 1,198 rows.




In [15]:
pct = ['approved_pct', 'bd_pct', 'td_pct']
for c in pct:
    df2[c + '_x_vol'] = df2[c] * df2.volume_mn

final = df2.groupby(['month', 'standard_name'], as_index=False).agg(
    volume_mn=('volume_mn', 'sum'),
    approved_x=('approved_pct_x_vol', 'sum'),
    bd_x=('bd_pct_x_vol', 'sum'),
    td_x=('td_pct_x_vol', 'sum'),
    is_ranked=('is_ranked', 'min'))

final['approved_pct'] = (final.approved_x / final.volume_mn).round(4)
final['bd_pct'] = (final.bd_x / final.volume_mn).round(4)
final['td_pct'] = (final.td_x / final.volume_mn).round(4)
final = final[['month', 'standard_name', 'volume_mn', 'approved_pct', 'bd_pct', 'td_pct', 'is_ranked']]
final = final.rename(columns={'standard_name': 'bank_name'})

print('rows:', len(final))
final[final.bank_name == 'Slice Small Finance Bank Limited'].tail(4)

rows: 1198


,month,bank_name,volume_mn,approved_pct,bd_pct,td_pct,is_ranked
1038,2026-04-01,Slice Small Finance Bank Limited,76.910000,94.2300,5.6700,0.1000,1
1088,2026-05-01,Slice Small Finance Bank Limited,85.940000,94.0000,5.9300,0.0700,1
1138,2026-06-01,Slice Small Finance Bank Limited,155.820000,90.8696,8.9773,0.1531,1
1187,2026-07-01,Slice Small Finance Bank Limited,99.923222,94.0900,5.8000,0.1100,1


## Step 14: Prove nothing changed in the totals

**What:** Compare each month's total volume and estimated failed payments before and after mapping. Also check for duplicates and the sum rule.

**Why:** Renaming and merging must not change any total. This is how we know the mapping is safe.




In [16]:
def monthly(d):
    d = d.copy()
    d['failed'] = d.volume_mn * d.td_pct / 100
    m = d.groupby('month').agg(volume_mn=('volume_mn', 'sum'), failed=('failed', 'sum'))
    m['weighted_td'] = m.failed / m.volume_mn * 100
    return m

before = monthly(df)
after = monthly(final)

assert np.allclose(before.volume_mn, after.volume_mn, atol=0.001)
assert np.allclose(before.failed, after.failed, atol=0.001)
print('monthly volume and estimated failed are unchanged by the merge')

assert not final.duplicated(['month', 'bank_name']).any()
dev = (final.approved_pct + final.bd_pct + final.td_pct - 100).abs().max()
print('max sum-rule deviation:', round(dev, 3))
print('banks per month:', sorted(set(final.groupby('month').size().tolist())))

after.loc[['2024-08-01', '2025-01-01', '2026-07-01']].round(3)

monthly volume and estimated failed are unchanged by the merge
max sum-rule deviation: 0.01
banks per month: [49, 50]


,volume_mn,failed,weighted_td
month,,,
2024-08-01,15498.84,54.888,0.354
2025-01-01,18004.92,90.542,0.503
2026-07-01,25897.80,103.225,0.399


## Step 15: Save the results

**What:** Save the mapping table and the mapped data, then download both.

**Why:** The mapping CSV is the audit trail for every rename. The mapped CSV is what we load into MySQL next.


In [17]:
mapping_out = mapping[['bank_raw', 'standard_name', 'rule', 'is_ranked', 'possible_same_as', 'note']] \
    .sort_values(['standard_name', 'bank_raw'])
mapping_out.to_csv('bank_mapping.csv', index=False)
final.to_csv('upi_remitter_mapped.csv', index=False)

print(pd.read_csv('bank_mapping.csv').shape, pd.read_csv('upi_remitter_mapped.csv').shape)

files.download('bank_mapping.csv')
files.download('upi_remitter_mapped.csv')

(117, 6) (1198, 7)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ✅ Summary

- 🔎 Found 117 spellings, cleaned to 67 with code, then to 61 standard banks after 6 evidence-based merges.
- 🧪 Tested suspects by checking whether two names ever appear in the same month and whether volume continues across the gap.
- 📝 Every decision is recorded in `bank_mapping.csv` with a rule and a note.
- 🔁 Merged Slice Small Finance Bank, listed twice in Mar'26 and Jun'26, by volume weighting.
- 📈 Result: 1,198 rows. Monthly volume and estimated failed payments are identical to before mapping.

⚠️ **Limits:** Three possible renames (Uttar Pradesh Gramin, Rajasthan Gramin, Andhra Pradesh Grameena) are kept separate because volume alone does not prove they are the same bank. IDFC First and the Andhra IMPS entry are merged on name and volume evidence, not on an NPCI statement.

➡️ **Next:** load into MySQL (`dim_bank` and `fact_upi_remitter`).